# MIH-Style Topic Relevance Pipeline (Custom Datasets + Keywords)

This notebook follows the same direct MIH workflow style:

`load -> keyword context -> LLM classify -> save kept/rejected + summary`

It is configured for your Nigeria data by default, but you can change paths in the config cell to run this on any dataset + keyword file.


In [1]:
from __future__ import annotations

import asyncio
import importlib.metadata
import json
import os
import random
import re
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import APIError, APITimeoutError, AsyncOpenAI, OpenAI, RateLimitError
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential_jitter
from tqdm.auto import tqdm

# ============================================================
# CONFIG (edit these for your own datasets + keywords)
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def find_root(start: Path) -> Path:
    start = start.resolve()
    for c in [start, *start.parents]:
        if (c / "Nigeria Audience Comments").exists() and (c / "Codebook and Keywords").exists():
            return c
    return start


ROOT = find_root(Path.cwd())
load_dotenv(ROOT / ".env")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY missing in .env")

# Input datasets (glob)
INPUT_GLOB = "Nigeria Audience Comments/**/*.xlsx"

# Keywords file + sheet
KEYWORDS_XLSX = ROOT / "Codebook and Keywords" / "NLC Proposed keywords.xlsx"
KEYWORDS_SHEET = "Nigeria"
KEYWORD_COL = "Keyword"
KEYWORD_RELEVANCE_COL = "Relevance to manosphere conversations"
KEYWORD_LEVELS_TO_KEEP = {"Highly relevant", "Moderately relevant"}

# Output + checkpoints
OUTPUT_DIR = ROOT / "Topic Relevant Comments - Nigeria - MIH Style"
CHECKPOINT_DIR = ROOT / "temp" / "mih_style_topic_relevance"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# LLM behavior (MIH-style direct classification)
MODEL_NAME = "gpt-4o-mini"
REVIEW_MODEL_NAME = "gpt-4o"  # optional second pass for borderlines
DO_SECOND_PASS_ON_BORDERLINE = False
MIN_CONFIDENCE_TO_KEEP = 0.55
CLASSIFY_SCOPE = "keyword_hits_only"  # "all" or "keyword_hits_only"

# Runtime tuning
CONCURRENCY = 3
CHECKPOINT_EVERY = 50
REQUEST_TIMEOUT_SECONDS = 20

# Optional throttle (MIH notebook style pacing)
SLEEP_BETWEEN_REQUESTS = 0.5

# Optional cheap smoke run (set to None for full run)
DRY_RUN_ROWS = None

# Rough pricing table for metrics only
PRICING_USD_PER_MTOK = {
    "gpt-4o-mini": {"input": 0.15, "cached_input": 0.075, "output": 0.60},
    "gpt-4o": {"input": 2.50, "cached_input": 1.25, "output": 10.00},
}

sync_client = OpenAI(api_key=OPENAI_API_KEY)
async_client = AsyncOpenAI(api_key=OPENAI_API_KEY)
RUN_START_TS = time.time()

print("Project root:", ROOT)
print("Input glob:", INPUT_GLOB)
print("Keywords:", KEYWORDS_XLSX, "| sheet:", KEYWORDS_SHEET)
print("Output dir:", OUTPUT_DIR)
print("Model:", MODEL_NAME, "| second pass:", DO_SECOND_PASS_ON_BORDERLINE, REVIEW_MODEL_NAME)


Project root: /Users/sushildalavi/Desktop/NLC/GATES-Project
Input glob: Nigeria Audience Comments/**/*.xlsx
Keywords: /Users/sushildalavi/Desktop/NLC/GATES-Project/Codebook and Keywords/NLC Proposed keywords.xlsx | sheet: Nigeria
Output dir: /Users/sushildalavi/Desktop/NLC/GATES-Project/Topic Relevant Comments - Nigeria - MIH Style
Model: gpt-4o-mini | second pass: False gpt-4o


In [2]:
def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    txt = str(value)
    txt = unicodedata.normalize("NFKC", txt)
    txt = txt.replace("‘", "'").replace("’", "'")
    txt = txt.replace("“", '"').replace("”", '"')
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt


def safe_model_tag(model: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", model)


def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def estimate_tokens(text: str) -> int:
    return max(1, int(len(text or "") / 4))


def llm_cost_usd(model: str, prompt_tokens: int, completion_tokens: int, cached_prompt_tokens: int = 0) -> float:
    pricing = PRICING_USD_PER_MTOK.get(model)
    if not pricing:
        return 0.0
    non_cached = max(int(prompt_tokens or 0) - int(cached_prompt_tokens or 0), 0)
    prompt_cost = (non_cached / 1_000_000) * pricing["input"]
    cached_cost = (int(cached_prompt_tokens or 0) / 1_000_000) * pricing.get("cached_input", pricing["input"])
    output_cost = (int(completion_tokens or 0) / 1_000_000) * pricing.get("output", 0.0)
    return round(prompt_cost + cached_cost + output_cost, 8)


def load_keywords() -> pd.DataFrame:
    df = pd.read_excel(KEYWORDS_XLSX, sheet_name=KEYWORDS_SHEET)
    missing = [c for c in [KEYWORD_COL, KEYWORD_RELEVANCE_COL] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing keyword columns in sheet '{KEYWORDS_SHEET}': {missing}")

    out = df[[KEYWORD_COL, KEYWORD_RELEVANCE_COL]].copy()
    out.columns = ["keyword", "relevance"]
    out["keyword"] = out["keyword"].map(normalize_text)
    out["relevance"] = out["relevance"].map(normalize_text)
    out = out[out["keyword"] != ""].copy()
    out = out[out["relevance"].isin(KEYWORD_LEVELS_TO_KEEP)].copy()
    out = out.drop_duplicates(subset=["keyword"]).reset_index(drop=True)
    return out


def compile_keyword_patterns(keywords: list[str]) -> list[tuple[str, re.Pattern[str]]]:
    patterns: list[tuple[str, re.Pattern[str]]] = []
    for kw in keywords:
        base = normalize_text(kw).lower().lstrip("#")
        if not base:
            continue
        escaped = re.escape(base).replace(r"\ ", r"\s+")
        pat = re.compile(rf"(?<!\w)#?{escaped}(?!\w)", flags=re.IGNORECASE)
        patterns.append((kw, pat))
    return patterns


def find_text_col(cols: list[str]) -> str:
    low = {str(c).strip().lower(): c for c in cols}
    for k in ["text", "comment", "comments", "content", "comment text"]:
        if k in low:
            return low[k]
    raise ValueError(f"No text/comment column found. Columns: {cols}")


def load_comment_files(input_glob: str) -> tuple[dict[str, pd.DataFrame], pd.DataFrame, pd.DataFrame]:
    files = sorted((ROOT / ".").glob(input_glob))
    files = [f for f in files if f.is_file() and not f.name.startswith("~$")]
    if not files:
        raise RuntimeError(f"No files found for glob: {input_glob}")

    file_tables: dict[str, pd.DataFrame] = {}
    master_rows: list[pd.DataFrame] = []
    inventory: list[dict[str, Any]] = []

    for path in files:
        raw = pd.read_excel(path)
        raw.columns = [str(c).strip() for c in raw.columns]

        rel_path = path.relative_to(ROOT)
        creator = path.parent.name
        video = path.stem
        source_file = rel_path.as_posix()

        df = raw.copy().reset_index(drop=True)
        df["source_row_index"] = df.index
        df["creator"] = creator
        df["video"] = video
        df["source_file"] = source_file
        df["source_file_stem"] = path.stem
        df["row_id"] = [f"{path.stem}#{i}" for i in df.index]

        text_col = find_text_col(list(df.columns))
        df["text_unified"] = df[text_col].map(normalize_text)

        if "author" not in df.columns:
            df["author"] = pd.NA
        if "likes" not in df.columns:
            df["likes"] = pd.NA

        if "replies" in df.columns:
            df["replies_unified"] = df["replies"]
        elif "reply_count" in df.columns:
            df["replies_unified"] = df["reply_count"]
        else:
            df["replies_unified"] = pd.NA

        df["url_unified"] = df["url"] if "url" in df.columns else pd.NA
        df["timestamp_unified"] = df["timestamp"] if "timestamp" in df.columns else pd.NA

        non_empty = df["text_unified"].str.strip() != ""
        df_non_empty = df.loc[non_empty].copy().reset_index(drop=True)

        file_tables[source_file] = df_non_empty

        master_rows.append(
            df_non_empty[
                [
                    "row_id",
                    "creator",
                    "video",
                    "source_file",
                    "source_file_stem",
                    "source_row_index",
                    "text_unified",
                ]
            ].rename(columns={"text_unified": "text"})
        )

        inventory.append(
            {
                "source_file": source_file,
                "creator": creator,
                "video": video,
                "raw_rows": int(len(df)),
                "non_empty_rows": int(len(df_non_empty)),
            }
        )

    master = pd.concat(master_rows, ignore_index=True)
    inv = pd.DataFrame(inventory).sort_values(["creator", "video"]).reset_index(drop=True)
    return file_tables, master, inv


KEYWORDS_DF = load_keywords()
KEYWORDS = KEYWORDS_DF["keyword"].tolist()
KEYWORD_PATTERNS = compile_keyword_patterns(KEYWORDS)

FILE_TABLES, MASTER_DF, INVENTORY_DF = load_comment_files(INPUT_GLOB)

print("Keyword count:", len(KEYWORDS_DF))
print("Comments (non-empty):", len(MASTER_DF))
print("Files:", len(INVENTORY_DF))
display(INVENTORY_DF)


Keyword count: 332
Comments (non-empty): 9735
Files: 10


,source_file,creator,video,raw_rows,non_empty_rows
0,Nigeria Audience Comments/Agba John Doe/Never ...,Agba John Doe,Never Leave Marriage Because Husband Cheated,511,511
1,Nigeria Audience Comments/Agba John Doe/Single...,Agba John Doe,Single Woman Earning Above 1.5M,245,245
2,Nigeria Audience Comments/Banky Wellington/Fin...,Banky Wellington,Final Say Faith,6389,6389
3,Nigeria Audience Comments/Banky Wellington/My ...,Banky Wellington,My Story Journey Through Hope and Faith,712,712
4,Nigeria Audience Comments/Deyemi Okanlawon/Men...,Deyemi Okanlawon,Men Must Hold Men Accountable,58,58
5,Nigeria Audience Comments/Deyemi Okanlawon/Sto...,Deyemi Okanlawon,Stop Raping Women Response,647,647
6,Nigeria Audience Comments/Shola/7 Women Will B...,Shola,7 Women Will Beg One Man to Marry,454,454
7,Nigeria Audience Comments/Shola/Women and Avai...,Shola,Women and Availability Trap,225,225
8,Nigeria Audience Comments/Wizarab/Child Suppor...,Wizarab,Child Support and Divorce,183,183
9,Nigeria Audience Comments/Wizarab/Sex Toys and...,Wizarab,Sex Toys and Raping Young Boys,311,311


In [3]:
def keyword_hits(text: str) -> list[str]:
    if not text:
        return []
    txt = normalize_text(text).lower()
    hits = [kw for kw, pat in KEYWORD_PATTERNS if pat.search(txt)]
    seen = set()
    out = []
    for h in hits:
        if h not in seen:
            seen.add(h)
            out.append(h)
    return out


MASTER_DF = MASTER_DF.copy()
MASTER_DF["kw_hits"] = [keyword_hits(t) for t in tqdm(MASTER_DF["text"], desc="Keyword matching")]
MASTER_DF["kw_hit_count"] = MASTER_DF["kw_hits"].map(len)

if CLASSIFY_SCOPE == "keyword_hits_only":
    CANDIDATE_DF = MASTER_DF.loc[MASTER_DF["kw_hit_count"] > 0].copy().reset_index(drop=True)
else:
    CANDIDATE_DF = MASTER_DF.copy().reset_index(drop=True)

if DRY_RUN_ROWS is not None:
    CANDIDATE_DF = CANDIDATE_DF.head(int(DRY_RUN_ROWS)).copy()

print("Classify scope:", CLASSIFY_SCOPE)
print("Rows to classify:", len(CANDIDATE_DF))
print("Keyword-hit rows in full data:", int((MASTER_DF["kw_hit_count"] > 0).sum()))
print("Keyword-hit rate (%):", round((MASTER_DF["kw_hit_count"] > 0).mean() * 100, 2))

per_file_hit = (
    MASTER_DF.assign(kw_hit=MASTER_DF["kw_hit_count"] > 0)
    .groupby(["creator", "video", "source_file"], as_index=False)
    .agg(non_empty_rows=("row_id", "count"), keyword_hit_rows=("kw_hit", "sum"))
)
per_file_hit["keyword_hit_rate_pct"] = (per_file_hit["keyword_hit_rows"] / per_file_hit["non_empty_rows"] * 100).round(2)
display(per_file_hit.sort_values(["creator", "video"]).reset_index(drop=True))


Keyword matching:   0%|          | 0/9735 [00:00<?, ?it/s]

Classify scope: keyword_hits_only
Rows to classify: 2840
Keyword-hit rows in full data: 2840
Keyword-hit rate (%): 29.17


,creator,video,source_file,non_empty_rows,keyword_hit_rows,keyword_hit_rate_pct
0,Agba John Doe,Never Leave Marriage Because Husband Cheated,Nigeria Audience Comments/Agba John Doe/Never ...,511,276,54.01
1,Agba John Doe,Single Woman Earning Above 1.5M,Nigeria Audience Comments/Agba John Doe/Single...,245,102,41.63
2,Banky Wellington,Final Say Faith,Nigeria Audience Comments/Banky Wellington/Fin...,6389,1664,26.04
3,Banky Wellington,My Story Journey Through Hope and Faith,Nigeria Audience Comments/Banky Wellington/My ...,712,156,21.91
4,Deyemi Okanlawon,Men Must Hold Men Accountable,Nigeria Audience Comments/Deyemi Okanlawon/Men...,58,22,37.93
5,Deyemi Okanlawon,Stop Raping Women Response,Nigeria Audience Comments/Deyemi Okanlawon/Sto...,647,234,36.17
6,Shola,7 Women Will Beg One Man to Marry,Nigeria Audience Comments/Shola/7 Women Will B...,454,142,31.28
7,Shola,Women and Availability Trap,Nigeria Audience Comments/Shola/Women and Avai...,225,63,28.00
8,Wizarab,Child Support and Divorce,Nigeria Audience Comments/Wizarab/Child Suppor...,183,62,33.88
9,Wizarab,Sex Toys and Raping Young Boys,Nigeria Audience Comments/Wizarab/Sex Toys and...,311,119,38.26


In [4]:
SYSTEM_PROMPT = """
You are a strict social-science coding assistant.

Task: classify whether a comment is topic-relevant to masculinity/manosphere/gender discourse.

Label definitions:
- yes: clear or reasonably direct engagement with masculinity, gender roles, male/female relationship dynamics, marriage norms, or gendered social expectations.
- borderline: weakly related, ambiguous, or missing enough context to be confidently yes/no.
- no: generic praise, spam, off-topic, or comments without meaningful gender/relationship-role content in this study context.

Return strict JSON only:
{
  "relevant": "yes|no|borderline",
  "confidence": 0.0,
  "reason": "one short sentence"
}
""".strip()


def build_prompt(row: dict[str, Any]) -> str:
    hits = row.get("kw_hits") or []
    kw_text = "; ".join(hits[:30]) if isinstance(hits, list) else str(hits)
    kw_text = kw_text if kw_text else "none"

    return f"""
Dataset context:
- Creator: {row.get('creator', '')}
- Video: {row.get('video', '')}
- Matched keywords: {kw_text}

Comment:
\"\"\"{row.get('text', '')}\"\"\"
""".strip()


def parse_json_obj(text: str) -> dict[str, Any]:
    text = (text or "").strip()
    if not text:
        return {}
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not m:
            return {}
        try:
            return json.loads(m.group(0))
        except Exception:
            return {}


def normalize_label(payload: dict[str, Any]) -> tuple[str, float, str]:
    lbl = str(payload.get("relevant", "borderline")).strip().lower()
    if lbl not in {"yes", "no", "borderline"}:
        lbl = "borderline"

    conf = payload.get("confidence", 0.5)
    try:
        conf = float(conf)
    except Exception:
        conf = 0.5
    conf = max(0.0, min(1.0, conf))

    reason = normalize_text(payload.get("reason", ""))
    if not reason:
        reason = "No reason provided."
    return lbl, conf, reason


def is_retryable(exc: BaseException) -> bool:
    return isinstance(exc, (RateLimitError, APITimeoutError, APIError, TimeoutError))


@retry(
    reraise=True,
    stop=stop_after_attempt(1),
    wait=wait_exponential_jitter(initial=1, max=5),
    retry=retry_if_exception(lambda exc: is_retryable(exc)),
)
async def classify_one(row: dict[str, Any], model: str) -> dict[str, Any]:
    resp = await async_client.chat.completions.create(
        model=model,
        temperature=0,
        timeout=REQUEST_TIMEOUT_SECONDS,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(row)},
        ],
    )

    raw = resp.choices[0].message.content or "{}"
    payload = parse_json_obj(raw)
    label, conf, reason = normalize_label(payload)

    usage = getattr(resp, "usage", None)
    pt = int(getattr(usage, "prompt_tokens", 0) or 0)
    ct = int(getattr(usage, "completion_tokens", 0) or 0)
    cached = 0
    if usage is not None:
        details = getattr(usage, "prompt_tokens_details", None)
        if details is not None:
            cached = int(getattr(details, "cached_tokens", 0) or 0)

    return {
        "row_id": row["row_id"],
        "llm_relevant": label,
        "llm_confidence": conf,
        "llm_reason": reason,
        "llm_raw": raw,
        "model_used": model,
        "prompt_tokens": pt,
        "completion_tokens": ct,
        "cached_prompt_tokens": cached,
        "llm_cost_usd": llm_cost_usd(model, pt, ct, cached),
        "classified_at_utc": now_utc_iso(),
    }


def checkpoint_path(stage: str, model: str) -> Path:
    return CHECKPOINT_DIR / f"{stage}_{safe_model_tag(model)}.parquet"


def load_checkpoint(stage: str, model: str) -> pd.DataFrame:
    p = checkpoint_path(stage, model)
    if not p.exists():
        return pd.DataFrame()
    df = pd.read_parquet(p)
    if "row_id" not in df.columns:
        return pd.DataFrame()
    return df.drop_duplicates(subset=["row_id"], keep="last").reset_index(drop=True)


def save_checkpoint(df: pd.DataFrame, stage: str, model: str) -> None:
    p = checkpoint_path(stage, model)
    p.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(p, index=False)


async def classify_dataframe(input_df: pd.DataFrame, model: str, stage: str) -> pd.DataFrame:
    if input_df.empty:
        return pd.DataFrame()

    existing = load_checkpoint(stage, model)
    done = set(existing["row_id"].tolist()) if not existing.empty else set()
    pending = input_df.loc[~input_df["row_id"].isin(done)].copy().reset_index(drop=True)

    print(f"{stage}: existing={len(done)}, pending={len(pending)}")
    if pending.empty:
        return existing

    sem = asyncio.Semaphore(CONCURRENCY)
    new_rows: list[dict[str, Any]] = []

    async def worker(row: dict[str, Any]) -> dict[str, Any]:
        async with sem:
            if SLEEP_BETWEEN_REQUESTS > 0:
                await asyncio.sleep(SLEEP_BETWEEN_REQUESTS)
            try:
                return await classify_one(row, model)
            except Exception as exc:
                return {
                    "row_id": row["row_id"],
                    "llm_relevant": "borderline",
                    "llm_confidence": 0.0,
                    "llm_reason": f"classifier_error: {type(exc).__name__}",
                    "llm_raw": str(exc),
                    "model_used": model,
                    "prompt_tokens": 0,
                    "completion_tokens": 0,
                    "cached_prompt_tokens": 0,
                    "llm_cost_usd": 0.0,
                    "classified_at_utc": now_utc_iso(),
                }

    tasks = [asyncio.create_task(worker(r)) for r in pending.to_dict(orient="records")]

    for i, fut in enumerate(tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=f"{stage}:{model}"), start=1):
        row_out = await fut
        new_rows.append(row_out)

        if i % CHECKPOINT_EVERY == 0 or i == len(tasks):
            cur = pd.DataFrame(new_rows)
            merged = pd.concat([existing, cur], ignore_index=True) if not existing.empty else cur
            merged = merged.sort_values("classified_at_utc").drop_duplicates(subset=["row_id"], keep="last")
            save_checkpoint(merged, stage, model)

    return load_checkpoint(stage, model)


In [5]:
# Stage 1: main pass
stage1 = await classify_dataframe(CANDIDATE_DF, MODEL_NAME, stage="stage1")
SCORING_DF = CANDIDATE_DF.merge(stage1, on="row_id", how="left")

# Stage 2 (optional): second pass only on borderlines / low-confidence
if DO_SECOND_PASS_ON_BORDERLINE:
    mask = SCORING_DF["llm_relevant"].eq("borderline") | (SCORING_DF["llm_confidence"].fillna(0.0) < MIN_CONFIDENCE_TO_KEEP)
    review_df = SCORING_DF.loc[mask].copy().reset_index(drop=True)
    print("Stage2 rows:", len(review_df))

    if not review_df.empty:
        stage2 = await classify_dataframe(review_df, REVIEW_MODEL_NAME, stage="stage2_review")
        stage2 = stage2.add_prefix("s2_").rename(columns={"s2_row_id": "row_id"})
        SCORING_DF = SCORING_DF.merge(stage2, on="row_id", how="left")
    else:
        SCORING_DF["s2_llm_relevant"] = pd.NA

    SCORING_DF["review_invoked"] = SCORING_DF["s2_llm_relevant"].notna()
    SCORING_DF["llm_relevant_final"] = np.where(SCORING_DF["review_invoked"], SCORING_DF["s2_llm_relevant"], SCORING_DF["llm_relevant"])
    SCORING_DF["llm_confidence_final"] = np.where(SCORING_DF["review_invoked"], SCORING_DF["s2_llm_confidence"], SCORING_DF["llm_confidence"])
    SCORING_DF["llm_reason_final"] = np.where(SCORING_DF["review_invoked"], SCORING_DF["s2_llm_reason"], SCORING_DF["llm_reason"])
    SCORING_DF["model_used_final"] = np.where(SCORING_DF["review_invoked"], SCORING_DF["s2_model_used"], SCORING_DF["model_used"])
    SCORING_DF["llm_cost_usd_final"] = np.where(SCORING_DF["review_invoked"], SCORING_DF["s2_llm_cost_usd"], SCORING_DF["llm_cost_usd"])
else:
    SCORING_DF["review_invoked"] = False
    SCORING_DF["llm_relevant_final"] = SCORING_DF["llm_relevant"]
    SCORING_DF["llm_confidence_final"] = SCORING_DF["llm_confidence"]
    SCORING_DF["llm_reason_final"] = SCORING_DF["llm_reason"]
    SCORING_DF["model_used_final"] = SCORING_DF["model_used"]
    SCORING_DF["llm_cost_usd_final"] = SCORING_DF["llm_cost_usd"]

SCORING_DF["final_yes"] = (
    SCORING_DF["llm_relevant_final"].eq("yes")
    & (SCORING_DF["llm_confidence_final"].fillna(0.0) >= MIN_CONFIDENCE_TO_KEEP)
)

print("Label distribution (final):")
print(SCORING_DF["llm_relevant_final"].value_counts(dropna=False).to_string())
print("Final kept rows:", int(SCORING_DF["final_yes"].sum()))
print("Total classified rows:", len(SCORING_DF))


stage1: existing=2840, pending=0
Label distribution (final):
llm_relevant_final
no            1755
yes            575
borderline     510
Final kept rows: 575
Total classified rows: 2840


In [6]:
def list_to_text(v: Any) -> str:
    if isinstance(v, list):
        return "; ".join(str(x) for x in v)
    if pd.isna(v):
        return ""
    return str(v)


extra_cols = [
    "row_id",
    "kw_hits",
    "kw_hit_count",
    "llm_relevant_final",
    "llm_confidence_final",
    "llm_reason_final",
    "model_used_final",
    "llm_cost_usd_final",
    "final_yes",
]

annot = SCORING_DF[extra_cols].copy()
annot["kw_hits"] = annot["kw_hits"].map(list_to_text)

summary_rows = []

for _, inv in INVENTORY_DF.iterrows():
    source_file = inv["source_file"]
    creator = inv["creator"]
    video = inv["video"]

    base = FILE_TABLES[source_file].copy()
    merged = base.merge(annot, on="row_id", how="left")

    merged["drop_stage"] = np.where(
        merged["final_yes"] == True,  # noqa: E712
        pd.NA,
        np.where(merged["llm_relevant_final"].isna(), "not_classified", "llm_no_or_low_conf"),
    )

    keep = merged.loc[merged["final_yes"] == True].copy()  # noqa: E712
    reject = merged.loc[merged["final_yes"] != True].copy()  # noqa: E712

    keep_path = OUTPUT_DIR / creator / f"{video}.xlsx"
    rej_path = OUTPUT_DIR / "_rejected" / creator / f"{video}.xlsx"
    keep_path.parent.mkdir(parents=True, exist_ok=True)
    rej_path.parent.mkdir(parents=True, exist_ok=True)

    keep.to_excel(keep_path, index=False)
    reject.to_excel(rej_path, index=False)

    classified_rows = int(merged["llm_relevant_final"].notna().sum())
    kept_rows = int(len(keep))
    total_cost = float(merged["llm_cost_usd_final"].fillna(0.0).sum())

    summary_rows.append(
        {
            "creator": creator,
            "video": video,
            "source_file": source_file,
            "raw_rows": int(inv["raw_rows"]),
            "non_empty_rows": int(inv["non_empty_rows"]),
            "classified_rows": classified_rows,
            "kept_rows": kept_rows,
            "kept_pct_of_non_empty": round(kept_rows / max(int(inv["non_empty_rows"]), 1) * 100, 2),
            "total_llm_cost_usd": round(total_cost, 6),
            "kept_output": keep_path.relative_to(ROOT).as_posix(),
            "rejected_output": rej_path.relative_to(ROOT).as_posix(),
        }
    )

SUMMARY_DF = pd.DataFrame(summary_rows).sort_values(["creator", "video"]).reset_index(drop=True)
summary_path = OUTPUT_DIR / "_pipeline_summary.xlsx"
SUMMARY_DF.to_excel(summary_path, index=False)

run_metadata = {
    "timestamp_utc": now_utc_iso(),
    "project_root": str(ROOT),
    "input_glob": INPUT_GLOB,
    "keywords_file": str(KEYWORDS_XLSX),
    "keywords_sheet": KEYWORDS_SHEET,
    "keyword_levels_kept": sorted(KEYWORD_LEVELS_TO_KEEP),
    "keyword_count": int(len(KEYWORDS_DF)),
    "classify_scope": CLASSIFY_SCOPE,
    "dry_run_rows": None if DRY_RUN_ROWS is None else int(DRY_RUN_ROWS),
    "model_primary": MODEL_NAME,
    "model_review": REVIEW_MODEL_NAME if DO_SECOND_PASS_ON_BORDERLINE else None,
    "min_confidence_to_keep": float(MIN_CONFIDENCE_TO_KEEP),
    "counts": {
        "non_empty_rows_total": int(INVENTORY_DF["non_empty_rows"].sum()),
        "classified_rows_total": int(SCORING_DF["llm_relevant_final"].notna().sum()),
        "kept_rows_total": int(SCORING_DF["final_yes"].sum()),
    },
    "costs_usd": {
        "total_llm_cost_usd": round(float(SCORING_DF["llm_cost_usd_final"].fillna(0.0).sum()), 6),
    },
    "runtime_seconds": round(time.time() - RUN_START_TS, 2),
    "library_versions": {
        "openai": importlib.metadata.version("openai"),
        "pandas": importlib.metadata.version("pandas"),
        "numpy": importlib.metadata.version("numpy"),
        "python-dotenv": importlib.metadata.version("python-dotenv"),
        "tenacity": importlib.metadata.version("tenacity"),
        "tqdm": importlib.metadata.version("tqdm"),
    },
}

meta_path = OUTPUT_DIR / "_run_metadata.json"
meta_path.write_text(json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8")

print("Saved outputs under:", OUTPUT_DIR)
print("Summary:", summary_path)
print("Metadata:", meta_path)


Saved outputs under: /Users/sushildalavi/Desktop/NLC/GATES-Project/Topic Relevant Comments - Nigeria - MIH Style
Summary: /Users/sushildalavi/Desktop/NLC/GATES-Project/Topic Relevant Comments - Nigeria - MIH Style/_pipeline_summary.xlsx
Metadata: /Users/sushildalavi/Desktop/NLC/GATES-Project/Topic Relevant Comments - Nigeria - MIH Style/_run_metadata.json


In [7]:
print("Pipeline summary:")
display(SUMMARY_DF)

creator_view = (
    SUMMARY_DF.groupby("creator", as_index=False)
    .agg(non_empty_rows=("non_empty_rows", "sum"), kept_rows=("kept_rows", "sum"), total_llm_cost_usd=("total_llm_cost_usd", "sum"))
)
creator_view["kept_pct"] = (creator_view["kept_rows"] / creator_view["non_empty_rows"] * 100).round(2)

print("\nRetention by creator:")
display(creator_view.sort_values("kept_pct", ascending=False).reset_index(drop=True))

print("\nFinal label counts:")
print(SCORING_DF["llm_relevant_final"].value_counts(dropna=False).to_string())

print("\nKept sample:")
display(SCORING_DF.loc[SCORING_DF["final_yes"] == True, ["creator", "video", "text", "llm_reason_final"]].head(12))  # noqa: E712

print("\nRejected sample:")
display(SCORING_DF.loc[SCORING_DF["final_yes"] != True, ["creator", "video", "text", "llm_relevant_final", "llm_reason_final"]].head(12))  # noqa: E712


Pipeline summary:


,creator,video,source_file,raw_rows,non_empty_rows,classified_rows,kept_rows,kept_pct_of_non_empty,total_llm_cost_usd,kept_output,rejected_output
0,Agba John Doe,Never Leave Marriage Because Husband Cheated,Nigeria Audience Comments/Agba John Doe/Never ...,511,511,276,191,37.38,0.0,Topic Relevant Comments - Nigeria - MIH Style/...,Topic Relevant Comments - Nigeria - MIH Style/...
1,Agba John Doe,Single Woman Earning Above 1.5M,Nigeria Audience Comments/Agba John Doe/Single...,245,245,102,81,33.06,0.0,Topic Relevant Comments - Nigeria - MIH Style/...,Topic Relevant Comments - Nigeria - MIH Style/...
2,Banky Wellington,Final Say Faith,Nigeria Audience Comments/Banky Wellington/Fin...,6389,6389,1664,64,1.00,0.0,Topic Relevant Comments - Nigeria - MIH Style/...,Topic Relevant Comments - Nigeria - MIH Style/...
3,Banky Wellington,My Story Journey Through Hope and Faith,Nigeria Audience Comments/Banky Wellington/My ...,712,712,156,3,0.42,0.0,Topic Relevant Comments - Nigeria - MIH Style/...,Topic Relevant Comments - Nigeria - MIH Style/...
4,Deyemi Okanlawon,Men Must Hold Men Accountable,Nigeria Audience Comments/Deyemi Okanlawon/Men...,58,58,22,8,13.79,0.0,Topic Relevant Comments - Nigeria - MIH Style/...,Topic Relevant Comments - Nigeria - MIH Style/...
5,Deyemi Okanlawon,Stop Raping Women Response,Nigeria Audience Comments/Deyemi Okanlawon/Sto...,647,647,234,38,5.87,0.0,Topic Relevant Comments - Nigeria - MIH Style/...,Topic Relevant Comments - Nigeria - MIH Style/...
6,Shola,7 Women Will Beg One Man to Marry,Nigeria Audience Comments/Shola/7 Women Will B...,454,454,142,59,13.00,0.0,Topic Relevant Comments - Nigeria - MIH Style/...,Topic Relevant Comments - Nigeria - MIH Style/...
7,Shola,Women and Availability Trap,Nigeria Audience Comments/Shola/Women and Avai...,225,225,63,31,13.78,0.0,Topic Relevant Comments - Nigeria - MIH Style/...,Topic Relevant Comments - Nigeria - MIH Style/...
8,Wizarab,Child Support and Divorce,Nigeria Audience Comments/Wizarab/Child Suppor...,183,183,62,41,22.40,0.0,Topic Relevant Comments - Nigeria - MIH Style/...,Topic Relevant Comments - Nigeria - MIH Style/...
9,Wizarab,Sex Toys and Raping Young Boys,Nigeria Audience Comments/Wizarab/Sex Toys and...,311,311,119,59,18.97,0.0,Topic Relevant Comments - Nigeria - MIH Style/...,Topic Relevant Comments - Nigeria - MIH Style/...



Retention by creator:


,creator,non_empty_rows,kept_rows,total_llm_cost_usd,kept_pct
0,Agba John Doe,756,272,0.0,35.98
1,Wizarab,494,100,0.0,20.24
2,Shola,679,90,0.0,13.25
3,Deyemi Okanlawon,705,46,0.0,6.52
4,Banky Wellington,7101,67,0.0,0.94



Final label counts:
llm_relevant_final
no            1755
yes            575
borderline     510

Kept sample:


,creator,video,text,llm_reason_final
0,Agba John Doe,Never Leave Marriage Because Husband Cheated,"For married women and especially in Africa, ne...",The comment discusses marriage dynamics and th...
1,Agba John Doe,Never Leave Marriage Because Husband Cheated,to leave. That man you leave can even find a v...,The comment discusses the implications of leav...
2,Agba John Doe,Never Leave Marriage Because Husband Cheated,Imagine once being a revered wife to a side wo...,The comment discusses the dynamics of marriage...
3,Agba John Doe,Never Leave Marriage Because Husband Cheated,Is it honestly favorable to you? Do you want t...,The comment discusses male-female relationship...
4,Agba John Doe,Never Leave Marriage Because Husband Cheated,do all that for you & love your children regar...,The comment discusses infidelity and marriage ...
5,Agba John Doe,Never Leave Marriage Because Husband Cheated,faithful husband is a disciplined husband but ...,The comment discusses gender differences in th...
6,Agba John Doe,Never Leave Marriage Because Husband Cheated,willing to replace her. For every woman a man ...,The comment discusses male behavior in the con...
7,Agba John Doe,Never Leave Marriage Because Husband Cheated,extreme exceptional cases. You can't pretend t...,The comment discusses body count in relation t...
8,Agba John Doe,Never Leave Marriage Because Husband Cheated,in the ways men think. Don't let social media ...,The comment discusses men's values regarding s...
10,Agba John Doe,Never Leave Marriage Because Husband Cheated,The cost-benefit analysis comes to play. If sh...,The comment discusses the implications of infi...



Rejected sample:


,creator,video,text,llm_relevant_final,llm_reason_final
9,Agba John Doe,Never Leave Marriage Because Husband Cheated,A girl I know was in a serious relationship wi...,borderline,The comment discusses a relationship dynamic b...
24,Agba John Doe,Never Leave Marriage Because Husband Cheated,Not only married women. Dating women too. The ...,borderline,Comment discusses dating dynamics but lacks cl...
30,Agba John Doe,Never Leave Marriage Because Husband Cheated,‍♀️‍♀️‍♀️ This marriage thing is not for the f...,borderline,The comment hints at marriage challenges but l...
32,Agba John Doe,Never Leave Marriage Because Husband Cheated,"Again I will say this, there's no manual for t...",borderline,The comment discusses relationships but lacks ...
41,Agba John Doe,Never Leave Marriage Because Husband Cheated,"the quick question here is that, are u current...",borderline,The comment questions faithfulness but lacks c...
44,Agba John Doe,Never Leave Marriage Because Husband Cheated,"I read the write up till the end, not all your...",borderline,The comment critiques views on women and marri...
46,Agba John Doe,Never Leave Marriage Because Husband Cheated,Is it all about servicing and getting serviced...,borderline,The comment raises questions about relationshi...
50,Agba John Doe,Never Leave Marriage Because Husband Cheated,If there is next world to come i prefer to com...,borderline,The comment expresses a preference for being a...
56,Agba John Doe,Never Leave Marriage Because Husband Cheated,This part is too direct. Oga Jon abeg put word...,no,The comment does not engage with masculinity o...
57,Agba John Doe,Never Leave Marriage Because Husband Cheated,"Well sometimes women just get tired. You see, ...",borderline,The comment touches on women's roles in marria...
